In [1]:
import sys
import os

python_path = sys.executable
os.environ['PYSPARK_PYTHON'] = python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = python_path

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [3]:
spark = (
    SparkSession.builder.appName('Pyspark_intro2')
    .master('local[4]')
    .config('spark.pyspark.python', python_path)
    .config('spark.pyspark.driver.python', python_path)
    .config('spark.python.use.daemon', 'false')
    .config('spark.python.worker.faulthandler.enabled', 'true')
    .getOrCreate()
)

In [5]:
df = spark.read.parquet('data/potato_prices_india.parquet')

### 1) basic inspection

In [5]:
df.limit(5).show()

+------------+---------+-------+------+-------+--------------+----------+--------------------+---------+---------+-----------+
|arrival_date|commodity|variety| grade|country|         state|  district|              market|min_price|max_price|modal_price|
+------------+---------+-------+------+-------+--------------+----------+--------------------+---------+---------+-----------+
|  2026-04-21|   potato| potato|   faq|  india|andhra pradesh|   chittor|           palamaner|   1000.0|   1200.0|     1100.0|
|  2026-04-21|   potato|  other|   faq|  india|         assam|    kamrup|     pamohi(garchuk)|   1300.0|   2200.0|     1500.0|
|  2026-04-21|   potato| potato|   faq|  india|         assam|    kamrup|     pamohi(garchuk)|    900.0|   1800.0|     1200.0|
|  2026-04-21|   potato| potato|   faq|  india|    chandigarh|chandigarh|chandigarh(grain/...|    100.0|    300.0|      300.0|
|  2026-04-21|   potato|  local|medium|  india|   chattisgarh|      durg|                durg|   1000.0|   1200

In [6]:
print(f'rows = {df.count()}')
print(f'cols = {len(df.columns)}')

rows = 1863539
cols = 11


In [7]:
list(df.schema)

[StructField('arrival_date', DateType(), True),
 StructField('commodity', StringType(), True),
 StructField('variety', StringType(), True),
 StructField('grade', StringType(), True),
 StructField('country', StringType(), True),
 StructField('state', StringType(), True),
 StructField('district', StringType(), True),
 StructField('market', StringType(), True),
 StructField('min_price', DoubleType(), True),
 StructField('max_price', DoubleType(), True),
 StructField('modal_price', DoubleType(), True)]

In [8]:
df.printSchema()

root
 |-- arrival_date: date (nullable = true)
 |-- commodity: string (nullable = true)
 |-- variety: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- market: string (nullable = true)
 |-- min_price: double (nullable = true)
 |-- max_price: double (nullable = true)
 |-- modal_price: double (nullable = true)



In [9]:
df.select('max_price', 'min_price', 'modal_price').describe().show()

+-------+------------------+-----------------+------------------+
|summary|         max_price|        min_price|       modal_price|
+-------+------------------+-----------------+------------------+
|  count|           1863539|          1863539|           1863539|
|   mean|1604.9050498379697|1345.570895817045|1486.1168374903878|
| stddev|  8883.83814729352|8286.281226605759| 8297.261246119935|
|    min|             150.0|             80.0|             150.0|
|    max|             1.2E7|           1.12E7|            1.12E7|
+-------+------------------+-----------------+------------------+



In [10]:
df.select('modal_price').summary().show()

+-------+------------------+
|summary|       modal_price|
+-------+------------------+
|  count|           1863539|
|   mean|1486.1168374903878|
| stddev| 8297.261246119935|
|    min|             150.0|
|    25%|             800.0|
|    50%|            1200.0|
|    75%|            1800.0|
|    max|            1.12E7|
+-------+------------------+



In [11]:
df.select('state', 'market', 'modal_price').withColumnRenamed('modal_price', 'cena_rynkowa').show()

+--------------+--------------------+------------+
|         state|              market|cena_rynkowa|
+--------------+--------------------+------------+
|andhra pradesh|           palamaner|      1100.0|
|         assam|     pamohi(garchuk)|      1500.0|
|         assam|     pamohi(garchuk)|      1200.0|
|    chandigarh|chandigarh(grain/...|       300.0|
|   chattisgarh|                durg|      1000.0|
|   chattisgarh|         rajnandgaon|       900.0|
|           goa|              mapusa|      1200.0|
|       gujarat|           kapadvanj|       750.0|
|       gujarat|ahmedabad(chimanb...|       925.0|
|       gujarat|ahmedabad(chimanb...|       725.0|
|       gujarat|             vadhvan|      1250.0|
|       gujarat|gondal(veg.market...|      1075.0|
|       gujarat|rajkot(veg.sub yard)|       895.0|
|       gujarat|     nadiyad(piplag)|      1200.0|
|       gujarat|anand(veg,yard,an...|       700.0|
|       gujarat|              nadiad|      1650.0|
|       gujarat|mehsana(mehsana

In [13]:
df.filter(F.col('state') == "assam").show()

+------------+---------+-------+-----+-------+-----+--------+--------------------+---------+---------+-----------+
|arrival_date|commodity|variety|grade|country|state|district|              market|min_price|max_price|modal_price|
+------------+---------+-------+-----+-------+-----+--------+--------------------+---------+---------+-----------+
|  2026-04-21|   potato|  other|  faq|  india|assam|  kamrup|     pamohi(garchuk)|   1300.0|   2200.0|     1500.0|
|  2026-04-21|   potato| potato|  faq|  india|assam|  kamrup|     pamohi(garchuk)|    900.0|   1800.0|     1200.0|
|  2026-04-20|   potato|  other|  faq|  india|assam| barpeta|        barpeta road|    900.0|   1100.0|     1000.0|
|  2026-04-20|   potato|  local|  faq|  india|assam|sonitpur|          dhekiajuli|    900.0|   1000.0|     1000.0|
|  2026-04-20|   potato| potato|  faq|  india|assam|  nagaon|           doomdumia|    900.0|   1100.0|     1000.0|
|  2026-04-20|   potato| potato|  faq|  india|assam|  nagaon|          haibargao

In [15]:
df.filter((F.col('state') == 'assam') & (F.col('modal_price') > 1200)).show()

+------------+---------+-------+-----+-------+-----+--------+---------------+---------+---------+-----------+
|arrival_date|commodity|variety|grade|country|state|district|         market|min_price|max_price|modal_price|
+------------+---------+-------+-----+-------+-----+--------+---------------+---------+---------+-----------+
|  2026-04-21|   potato|  other|  faq|  india|assam|  kamrup|pamohi(garchuk)|   1300.0|   2200.0|     1500.0|
|  2026-04-19|   potato|  other|  faq|  india|assam|  kamrup|pamohi(garchuk)|   1300.0|   2000.0|     1500.0|
|  2026-04-18|   potato|  other|  faq|  india|assam|  kamrup|pamohi(garchuk)|   1300.0|   2200.0|     1500.0|
|  2026-04-17|   potato|  other|  faq|  india|assam|  kamrup|pamohi(garchuk)|   1300.0|   2200.0|     1500.0|
|  2026-04-16|   potato|  other|  faq|  india|assam|  kamrup|pamohi(garchuk)|   1300.0|   2200.0|     1500.0|
|  2026-04-14|   potato|  other|  faq|  india|assam|  kamrup|pamohi(garchuk)|   1300.0|   2000.0|     1600.0|
|  2026-04

In [17]:
df.filter(F.col('state').isin('assam', 'chattisgarh')).show()

+------------+---------+-------+-------+-------+-----------+-----------+--------------------+---------+---------+-----------+
|arrival_date|commodity|variety|  grade|country|      state|   district|              market|min_price|max_price|modal_price|
+------------+---------+-------+-------+-------+-----------+-----------+--------------------+---------+---------+-----------+
|  2026-04-21|   potato|  other|    faq|  india|      assam|     kamrup|     pamohi(garchuk)|   1300.0|   2200.0|     1500.0|
|  2026-04-21|   potato| potato|    faq|  india|      assam|     kamrup|     pamohi(garchuk)|    900.0|   1800.0|     1200.0|
|  2026-04-21|   potato|  local| medium|  india|chattisgarh|       durg|                durg|   1000.0|   1200.0|     1000.0|
|  2026-04-21|   potato| potato|grade b|  india|chattisgarh|rajnandgaon|         rajnandgaon|    800.0|   1000.0|      900.0|
|  2026-04-20|   potato|  other|    faq|  india|      assam|    barpeta|        barpeta road|    900.0|   1100.0|     

In [19]:
df.filter(F.col('market').contains('grain')).show()

+------------+---------+-------+-------+-------+-------------+----------+--------------------+---------+---------+-----------+
|arrival_date|commodity|variety|  grade|country|        state|  district|              market|min_price|max_price|modal_price|
+------------+---------+-------+-------+-------+-------------+----------+--------------------+---------+---------+-----------+
|  2026-04-21|   potato| potato|    faq|  india|   chandigarh|chandigarh|chandigarh(grain/...|    100.0|    300.0|      300.0|
|  2026-04-20|   potato|  other|    faq|  india|uttar pradesh|    kanpur|       kanpur(grain)|    600.0|    600.0|      600.0|
|  2026-04-18|   potato| potato|    faq|  india|   chandigarh|chandigarh|chandigarh(grain/...|    100.0|    300.0|      300.0|
|  2026-04-18|   potato|  other|    faq|  india|uttar pradesh|    kanpur|       kanpur(grain)|    600.0|    600.0|      600.0|
|  2026-04-17|   potato| potato|    faq|  india|   chandigarh|chandigarh|chandigarh(grain/...|    100.0|    300

In [21]:
df.filter(F.col('modal_price').between(1000, 2500)).show()

+------------+---------+-----------+-------+-------+----------------+----------------+--------------------+---------+---------+-----------+
|arrival_date|commodity|    variety|  grade|country|           state|        district|              market|min_price|max_price|modal_price|
+------------+---------+-----------+-------+-------+----------------+----------------+--------------------+---------+---------+-----------+
|  2026-04-21|   potato|     potato|    faq|  india|  andhra pradesh|         chittor|           palamaner|   1000.0|   1200.0|     1100.0|
|  2026-04-21|   potato|      other|    faq|  india|           assam|          kamrup|     pamohi(garchuk)|   1300.0|   2200.0|     1500.0|
|  2026-04-21|   potato|     potato|    faq|  india|           assam|          kamrup|     pamohi(garchuk)|    900.0|   1800.0|     1200.0|
|  2026-04-21|   potato|      local| medium|  india|     chattisgarh|            durg|                durg|   1000.0|   1200.0|     1000.0|
|  2026-04-21|   pot

In [22]:
df.withColumn('price_spread', F.col('max_price') - F.col('min_price')).show(5)

+------------+---------+-------+------+-------+--------------+----------+--------------------+---------+---------+-----------+------------+
|arrival_date|commodity|variety| grade|country|         state|  district|              market|min_price|max_price|modal_price|price_spread|
+------------+---------+-------+------+-------+--------------+----------+--------------------+---------+---------+-----------+------------+
|  2026-04-21|   potato| potato|   faq|  india|andhra pradesh|   chittor|           palamaner|   1000.0|   1200.0|     1100.0|       200.0|
|  2026-04-21|   potato|  other|   faq|  india|         assam|    kamrup|     pamohi(garchuk)|   1300.0|   2200.0|     1500.0|       900.0|
|  2026-04-21|   potato| potato|   faq|  india|         assam|    kamrup|     pamohi(garchuk)|    900.0|   1800.0|     1200.0|       900.0|
|  2026-04-21|   potato| potato|   faq|  india|    chandigarh|chandigarh|chandigarh(grain/...|    100.0|    300.0|      300.0|       200.0|
|  2026-04-21|   pot

In [25]:
df.withColumn('price_category', F.when(
    F.col('modal_price') < 500, 'cheap')
    .when((F.col('modal_price') >= 500) & (F.col('modal_price') < 2000), 'average')
    .when(F.col('modal_price') >= 2000, 'high')
).limit(5).show()

+------------+---------+-------+------+-------+--------------+----------+--------------------+---------+---------+-----------+--------------+
|arrival_date|commodity|variety| grade|country|         state|  district|              market|min_price|max_price|modal_price|price_category|
+------------+---------+-------+------+-------+--------------+----------+--------------------+---------+---------+-----------+--------------+
|  2026-04-21|   potato| potato|   faq|  india|andhra pradesh|   chittor|           palamaner|   1000.0|   1200.0|     1100.0|       average|
|  2026-04-21|   potato|  other|   faq|  india|         assam|    kamrup|     pamohi(garchuk)|   1300.0|   2200.0|     1500.0|       average|
|  2026-04-21|   potato| potato|   faq|  india|         assam|    kamrup|     pamohi(garchuk)|    900.0|   1800.0|     1200.0|       average|
|  2026-04-21|   potato| potato|   faq|  india|    chandigarh|chandigarh|chandigarh(grain/...|    100.0|    300.0|      300.0|         cheap|
|  202

In [33]:
df.withColumn('date_parsed', F.to_date('arrival_date')).withColumn(
    'year', F.year('date_parsed')).withColumn(
    'month', F.month('date_parsed')).withColumn(
    'day', F.day('date_parsed')).withColumn(
    'quarter', F.quarter(F.col('date_parsed'))).limit(5).show()

+------------+---------+-------+------+-------+--------------+----------+--------------------+---------+---------+-----------+-----------+----+-----+---+-------+
|arrival_date|commodity|variety| grade|country|         state|  district|              market|min_price|max_price|modal_price|date_parsed|year|month|day|quarter|
+------------+---------+-------+------+-------+--------------+----------+--------------------+---------+---------+-----------+-----------+----+-----+---+-------+
|  2026-04-21|   potato| potato|   faq|  india|andhra pradesh|   chittor|           palamaner|   1000.0|   1200.0|     1100.0| 2026-04-21|2026|    4| 21|      2|
|  2026-04-21|   potato|  other|   faq|  india|         assam|    kamrup|     pamohi(garchuk)|   1300.0|   2200.0|     1500.0| 2026-04-21|2026|    4| 21|      2|
|  2026-04-21|   potato| potato|   faq|  india|         assam|    kamrup|     pamohi(garchuk)|    900.0|   1800.0|     1200.0| 2026-04-21|2026|    4| 21|      2|
|  2026-04-21|   potato| pot

In [36]:
df.withColumn(
    'state_upper', F.upper(F.col('state'))) \
    .withColumn('full_location', F.concat_ws(' -> ', F.col('state_upper'), F.col('district'), F.col('market'))) \
    .select('full_location', 'modal_price').limit(5).show()

+--------------------+-----------+
|       full_location|modal_price|
+--------------------+-----------+
|ANDHRA PRADESH ->...|     1100.0|
|ASSAM -> kamrup -...|     1500.0|
|ASSAM -> kamrup -...|     1200.0|
|CHANDIGARH -> cha...|      300.0|
|CHATTISGARH -> du...|     1000.0|
+--------------------+-----------+



In [37]:
df.groupBy('state').agg(
    F.countDistinct('district').alias('unique districts')
).show()

+-----------------+----------------+
|            state|unique districts|
+-----------------+----------------+
|      maharashtra|              21|
|        meghalaya|              10|
|           odisha|              30|
|          haryana|              21|
|      west bengal|              24|
|              goa|               1|
|jammu and kashmir|              11|
|           punjab|              22|
|        karnataka|              23|
|   andhra pradesh|               6|
|        telangana|               9|
|         nagaland|              11|
|            bihar|              39|
|   madhya pradesh|              39|
|        jharkhand|              21|
|            assam|              20|
|           kerala|              12|
|       tamil nadu|              36|
| himachal pradesh|              10|
|       chandigarh|               1|
+-----------------+----------------+
only showing top 20 rows


In [7]:
df.groupBy('state').agg(
    F.count('market').alias('total_markets'),
    F.round(F.avg('modal_price'), 2).alias('avg_modal_price'),
    F.max('max_price').alias('highest_price')
).orderBy(F.desc('avg_modal_price')).show()

+-------------------+-------------+---------------+-------------+
|              state|total_markets|avg_modal_price|highest_price|
+-------------------+-------------+---------------+-------------+
|andaman and nicobar|          416|        4398.08|       8000.0|
|         tamil nadu|        71079|        4308.84|      77000.0|
|           nagaland|         4058|        3427.78|      35000.0|
|             kerala|       120994|        3230.33|        1.2E7|
|            mizoram|          579|        2865.42|      10000.0|
|          meghalaya|         4022|        2265.17|      60000.0|
|            manipur|         6572|        2228.37|       8000.0|
|            tripura|        48292|        2224.57|      40000.0|
|          telangana|        18982|         1946.7|     190000.0|
|  arunachal pradesh|          248|        1930.22|       6000.0|
|          karnataka|        32819|        1746.38|      35000.0|
|             odisha|       114321|        1663.68|      25000.0|
|     andh

In [12]:
df.groupBy('state').pivot('variety').agg(
    F.round(F.avg('modal_price'), 2)
).show()

+-----------------+-------+-------+------------+-------+-------+-------+-----------+--------+--------+-------+------+-------------+----------------+-----------+-------+----------------+-------+-------+--------------+-------+-----------+-------+-------+
|            state|badshah|    big|chandermukhi|  chips|   desi| f.a.q.|great scott|haldwani|jalander|  jyoti| kuber|kufri giriraj|kufri khasi_garo|kufri megha|  local|military special|  other| potato|potato-organic|    red|red nanital| shimla|sinduri|
+-----------------+-------+-------+------------+-------+-------+-------+-----------+--------+--------+-------+------+-------------+----------------+-----------+-------+----------------+-------+-------+--------------+-------+-----------+-------+-------+
|      maharashtra|   NULL|   NULL|        NULL|   NULL|   NULL|   NULL|       NULL|    NULL|    NULL|   NULL|  NULL|         NULL|            NULL|       NULL|1653.88|            NULL|1409.23|   NULL|          NULL|   NULL|       NULL|   NU

In [ ]:
window_state = Window.partitionBy("state").orderBy(F.desc("modal_price"))

df.withColumn("rank_in_state", F.dense_rank().over(window_state)) \
  .filter(F.col("rank_in_state") == 1) \
  .select("state", "market", "modal_price", "rank_in_state") \
  .show()

In [15]:
window_state = Window.partitionBy('state').orderBy(F.desc('modal_price'))

df.withColumn('rank_in_state', F.dense_rank().over(window_state)) \
.filter(F.col('rank_in_state') == 1) \
.select('state', 'market', 'modal_price', 'rank_in_state') \
.show()

+-------------------+--------------------+-----------+-------------+
|              state|              market|modal_price|rank_in_state|
+-------------------+--------------------+-----------+-------------+
|andaman and nicobar|            diglipur|     7000.0|            1|
|andaman and nicobar|          port blair|     7000.0|            1|
|     andhra pradesh|           palamaner|     4000.0|            1|
|  arunachal pradesh|          naharlagun|     5000.0|            1|
|              assam|               bijni|    14000.0|            1|
|              assam|               bijni|    14000.0|            1|
|              bihar|            jainagar|    20000.0|            1|
|              bihar|            jainagar|    20000.0|            1|
|              bihar|            jainagar|    20000.0|            1|
|         chandigarh|chandigarh(grain/...|     2500.0|            1|
|         chandigarh|chandigarh(grain/...|     2500.0|            1|
|         chandigarh|chandigarh(gr

In [16]:
window_avg = Window.partitionBy("state")

df.withColumn("state_avg_price", F.avg("modal_price").over(window_avg)) \
  .withColumn("diff_from_avg", F.col("modal_price") - F.col("state_avg_price")) \
  .select("state", "market", "modal_price", "diff_from_avg") \
  .show(5)

+-------+--------------+-----------+-------------------+
|  state|        market|modal_price|      diff_from_avg|
+-------+--------------+-----------+-------------------+
|haryana|         sohna|     1000.0|-30.792941746954057|
|haryana|         tauru|      700.0|-330.79294174695406|
|haryana|        pundri|      850.0|-180.79294174695406|
|haryana|barwala(hisar)|      700.0|-330.79294174695406|
|haryana|    mustafabad|      700.0|-330.79294174695406|
+-------+--------------+-----------+-------------------+
only showing top 5 rows


In [17]:
df.createOrReplaceTempView("agri_data")

spark.sql("""
    SELECT 
        state,
        COUNT(DISTINCT market) AS markets_count,
        ROUND(AVG(modal_price), 2) AS avg_price
    FROM agri_data
    WHERE grade = 'faq'
    GROUP BY state
    HAVING avg_price > 1000
    ORDER BY avg_price DESC
""").show()

+-------------------+-------------+---------+
|              state|markets_count|avg_price|
+-------------------+-------------+---------+
|andaman and nicobar|            3|  4398.08|
|           nagaland|           16|  3433.85|
|             kerala|           77|  3221.98|
|            mizoram|            4|  2866.05|
|            tripura|           34|  2241.82|
|          meghalaya|           15|  2234.09|
|            manipur|            5|  2226.68|
|          telangana|           27|  2153.56|
|  arunachal pradesh|            3|  1930.22|
|          karnataka|           53|  1747.96|
|     andhra pradesh|            2|  1716.45|
|             odisha|           94|  1655.41|
|                goa|            1|  1600.71|
|              bihar|           91|   1519.0|
|   himachal pradesh|           49|  1486.83|
|  jammu and kashmir|           16|  1457.09|
|        chattisgarh|            5|  1454.12|
|        maharashtra|           62|  1451.38|
|        west bengal|           73